## Feature 1 Test

<p>The major objectives that need to be solved during this feature 1 are as follows:</p>
<ol>
<li>This feature mainly plans the trip for the user from his desired location to destination</li>
<li>This mainly consists of inputs such as Date, Time, From, To</li>
<li>The system analyzes either it is an local visit or national visit or international visit through the inputs of location and destination</li>
<li>In the desired location, the user can enter the location where he start instead of location of nearest airport, or selecting the nearest railway station and same for the destination address</li>
<li>Based on the input retreived, the system performes web searching for the plans to make the user reach from source to destination, from them it will select the optimized plan based on cost, time of travel</li>
</ol>

In [1]:
# importing necessary modules
from selenium import webdriver
from bs4 import BeautifulSoup
import os,os.path
import pandas as pd

In [2]:
# first let's create an Edge to take the address from the user
map_browser = webdriver.Edge()
link = 'https://www.google.com/maps'

In [3]:
# Animating Code
def animate_loading(base_text, step_count):
    """Overwrites the current terminal line with 1 to 4 animated dots."""
    dots = '.' * ((step_count % 4) + 1)
    # \r resets the cursor to the beginning of the line, spaces clear leftover trailing dots
    print(f'\r{base_text}{dots:<4}', end='', flush=True)

In [ ]:
import time
import keyboard

try:
    print('Select the start address from the map!')
    map_browser.get(link)
    time.sleep(2)

    # 1. Corrected page_source (no parentheses)
    source = map_browser.page_source
    soup = BeautifulSoup(source, 'html.parser')

    # 2. Extract title text
    title = soup.find('title').text if soup.find('title') else ""

    # 3. Fixed loop (removed map_browser.get inside loop so the page doesn't reset)
    step = 0
    while title.strip().lower() == 'google maps':
        animate_loading('Waiting for starting location selection from maps',step)
        step += 1
        time.sleep(2)

        # Access page_source directly from map_browser without ()
        source = map_browser.page_source
        soup = BeautifulSoup(source, 'html.parser')
        title = soup.find('title').text if soup.find('title') else ""

    print()

    # 4. Use map_browser to click (not start_map)
    map_browser.find_element('css selector', 'button.S9kvJb').click()
    time.sleep(2)


    # now we need to swap the chosen location to starting location since it is like showing as destination when selected at first

    button_div = map_browser.find_element('css selector','div.OFHA4')

    button_div.find_element('css selector','button.j9zajd').click()
    time.sleep(5)

    # now upto here the location got swapped to the first making as starting address

    # # now we need to click the button add destination

    # add_destination_div = map_browser.find_element('css selector','div.JuLCid kA9KIf')
    # add_dest_button = add_destination_div.find_element('css selector','button.fC7rrc xiw3Pd').click()

    # destination part 2: add directly from your location instead of clicking add destination

    # adding destination

    input_dest = map_browser.find_elements('css selector','input.ZBTq6e')
    input_dest[1].click()

    # again asking to give the destination

    print(f'Select the Destination from the maps!')
    # Wait until the user selects a destination (value is not blank and not placeholder text)
    placeholder = 'Choose destination, or click on the map...'
    step = 0
    while True:
        # Re-fetch the elements to ensure fresh DOM references
        current_dest = map_browser.find_elements('css selector', 'input.ZBTq6e')[1]
        dest_val = current_dest.get_attribute('aria-label') or ""

        if dest_val.strip() != "" and dest_val != placeholder:
            break

        time.sleep(2)
        animate_loading('Waiting for destination selection from maps',step)
        step += 1

    print()
    # After selecting the destination, Lets project the details

    # Retreive the details finally after selecting, upto here it is working and now we need to retreive the details of from and to address

    # Retreiving the start address from the chosen map
    time.sleep(10)
    input_elements = map_browser.find_elements('css selector','input.ZBTq6e')

    input_elements[0].click()

    start_address = input_elements[0].get_attribute('aria-label')
    input_elements[1].click()
    dest_address = input_elements[1].get_attribute('aria-label')

    print('-----------------------------User Pinging.------------------------------------')

    print(f'Boarding Location: {start_address}')
    print(f'Arrival Location: {dest_address}')

    print('-'*80)


    keyboard.wait('ctrl+q')
    for i in 'Closing browser.....':
        print(f'{i}',end='',flush = True)
        time.sleep(0.07)


except Exception as e:
    print(f'Failed due to error: {e}')

finally:

    map_browser.quit()

Select the start address from the map!
Waiting for starting location selection from maps..  
Select the Destination from the maps!
Waiting for destination selection from maps..  
-----------------------------User Pinging.------------------------------------
Boarding Location: Starting point Nila Hostel B Block IIIT Kottayam, QM35+H33, Nechipuzhoor, Keralam 686635
Arrival Location: Destination Kambhampadu, Andhra Pradesh 521227
--------------------------------------------------
Closing browser.....

In [6]:
# now we need to get the user
start_address, dest_address

('Starting point Nila Hostel B Block IIIT Kottayam, QM35+H33, Nechipuzhoor, Keralam 686635',
 'Destination Kambhampadu, Andhra Pradesh 521227')

In [9]:
start_point = start_address[14::].strip()
end_point = dest_address[11::].strip()

start_point,end_point

('Nila Hostel B Block IIIT Kottayam, QM35+H33, Nechipuzhoor, Keralam 686635',
 'Kambhampadu, Andhra Pradesh 521227')

## Input Block Ahead

In [ ]:
# Now from here, i need to find the best path to go from start_point to end_point with the information that user gave to us

# date of travel, time of travel, budget
import re

date_of_travel = input('Enter the date of travel (DD/MM/YYYY): ')
date_pattern = r'^(?:(?:0[1-9]|[12][0-9]|3[01])/(?:0[13578]|1[02])|(?:0[1-9]|[12][0-9]|30)/(?:0[469]|11)|(?:0[1-9]|1[0-9]|2[0-8])/02)/(?:19|20)\d\d$|^(?:29/02/(?:(?:19|20)(?:0[48]|[2468][048]|[13579][26])|(?:1600|2000)))$'

# testing date_pattern

# print(bool(re.match(date_pattern,'29/02/2025')))

while in_valid := bool(re.match(date_pattern,date_of_travel)) == False:
  date_of_travel = input('Enter a valid date of travel (DD/MM/YYYY): ')

# now lets take the budget of the user
pattern = r'^(?:[01][0-9]|2[0-3]):[0-5][0-9]$'
time_of_travel = input('Enter the start time of the trip (in 24 hour format with (AM/PM)): ')
while in_valid := bool(re.match(pattern,time_of_travel[:-3:])) == False:
  time_of_travel = input('Enter the start time  of the trip correctly (in 24 hour format with (AM/PM)): ')
# budget = input('Enter the budget of the trip in Rupees: ₹') # lets remove the budget for the sake of time and lets try for getting more than one solution with respect to cost


print('---------------------------------Your Trip Summary------------------------------------------------------------')

print(f'Start Point    :=             {start_point}')
print(f'End Point      :=             {end_point}')
print(f'Time of Start  :=             {time_of_travel}')
print(f'Date of Travel :=             {date_of_travel}')
# print(f'Budget         :=             ₹{budget}')

print('-'*110)

---------------------------------Your Trip Summary------------------------------------------------------------
Start Point    :=             Nila Hostel B Block IIIT Kottayam, QM35+H33, Nechipuzhoor, Keralam 686635
End Point      :=             Kambhampadu, Andhra Pradesh 521227
Time of Start  :=             07:00 AM
Date of Travel :=             16/11/2026
Budget         :=             ₹5500
--------------------------------------------------------------------------------------------------------------


In [28]:
# now lets try for all options that makes the trip successfully within the specified time, specified date and find more than one solution and order them by travel cost